# CELESTE Walkthrough

This tutorial assumes have installed CELESTE and its requirement packages and illustrates how to use the CELESTE library to develop your own unit and system models.
It aims to help you decode the library and get you up and running with the models quickly.

# Part 1 - Overview of the CELESTE Framework

The Framework is based on 3 core modules:
- base
- thermo
- systems

Part 2 will focus on making components based on Base and Thermo, while Part 3 will focus on Systems. 
The tutorial will be extended to include different related topics.

These core modules are then used to build unit and system models to form the CELESTE Standard Library (CSL) consisting of:
- components - Process components i.e. compressors and GPU
- electrochem - electrochemical reactor models i.e. SOC and AEL
- flow - flow control objects i.e. sources, sinks and mixers
- heat - Heat transfer objects i.e. HEX and Heaters
- interfaces - information transfer between objects i.e. Flange
- media - Fluid properties for specific mixtures using thermo
- reactors - Non-electrochemical reactor models i.e. methanation, HB etc.

# Part 2 - Components

These models in the CSL contain a specific structure given by ``ModelBase``

All unit models follow the structure:

```python
class UnitModel(ModelBase):
    def __init__(self, name, Medium, [other parameters]):
        super().__init__(locals())
        [initialise interfaces]

    def Eqs(self):
        self.Eq = [system of equations]
        return self.Eq

    def SomeOtherMethod(self): 
        return SomeOtherResult

    @property
    def someProperty
        [possible execution code here]
        return someValue

    def f_sol(self, x): 
        [unit variables that are unknown] = x[]

    def run_sol(self):
        x0 = [initial guess for unknowns]
        sol = solver(fun=self.f_sol, guess=x0)
        return sol

```

Here, `__init__()` and `Eqs()` are also found in `ModelBase` but were mostly empty and we add `SomeOtherMethod()`, `f_sol()`, and `run_sol()` on top of this to give the behaviour we want to model.

Lets look closer at what should be done for each method with the example of a mixer. We can Illustrate how a mixer should work as follows.
![mixer figure](Data/Mixer.png)

We have three streams, 2 inlet and one outlet therefore we need three flanges. 

```python

class Mixer(ModelBase):
    def __init__(self, name, Medium, [params to be defined]):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, [params F0])
        self.F1 = Flange(name + '.F1', Medium=Medium, [params F1])
        self.F2 = Flange(name + '.F2', Medium=Medium, [params F2])
```





Here we are using ModelBase and Flange which come from the CELESTE library and must be imported before mentioned:

In [1]:
from celeste.CSL import ModelBase
from celeste.CSL.interfaces import Flange
    

## Flange

Lets take a detour and zoom into the flange.

The Flange object takes the params `Flange(name, Medium, n, Xi, T, P)` 
- For `name` we related this to the name of the unit to give us `"mixer.F0"` as the name of the F0 flange.
- For the `Medium`, this comes from the `media` module described below
- `n` is the molar flow rate
- `Xi` the molar composition of each species
- `T` the temperature at the flange
- `P` the pressure at the flange


This is used in almost every object so its good to get comfortable with this object. Lets make a Flange object using the Air Medium:

In [2]:
import celeste.CSL.media as media
import numpy as np
f = Flange('f', Medium=media.Air(), n=1, Xi=np.array([0.21, 0.79]), T=300, P=1) # Normal Air flowing 1 mol/s @300K, 1bar

We can view the main detials of this flange by printing it:

In [3]:
print(f)

n /mol/s          1.000000
m /g/s           28.850334
T /K            300.000000
P /bar            1.000000
h /kJ/mol         0.053981
cp /kJ/mol/K      0.029180
Xi_O2             0.210000
Xi_N2             0.790000


This returns a pandas DataFrame of the impotant flow details.
We can also look into some thermodynamic properties such as enthalpy:

In [4]:
f.h #kJ/mol

np.float64(0.05398080388566671)

Other properties include:
- P: Pressure /bar
- T: Temperature /K
- Xi: Mole Fraction /-
- Yi: Mass Fraction /-
- cp: Heat Capacity /kJ/mol/K
- g: Gibbs Free Energy /kJ/mol
- h: Enthalpy /kJ/mol
- mw: Molar Mass /g/mol
- s: Entropy /kJ/mol/K

We can also change the T, P, and/or Xi of the flange after initialising it and it will automatically recalculate the properties:

In [5]:
f.T = 900
f.h

np.float64(18.437032884433542)

In this way the flange not only becomes and interfaces between objects, but also one for conveniently calculating thermodynamic properties for specific mixtures based on the media object.

## Media

Let's continue the detour and quickly look into the media package.
The core CSL contains the following media:

- ``Medium_Base(name, species)`` - Contains the core infrastructure for all media models
- ``Medium_NASAbase(name, species, T, P, Xi)`` - Contains core infrastructure for media models using NASA polynomial thermo properties
    - ``Air['O2','N2']`` - Simple Air model
    - ``Fuel_CH4['H2', 'CH4', 'CO2', 'CO', 'H2O', 'N2']`` - For SOC and Downstream processes
    - ``Fuel_CH4L['H2', 'CH4', 'CO2', 'CO', 'H2O(L)', 'N2']`` - For considering condensed water phase
    - ``Fuel_MeOH['H2', 'CH4', 'CO2', 'CO', 'H2O', 'N2', 'CH3OH']`` - For Methanol process
    - ``HB['H2','N2','NH3']`` - For Haber-Bosch process
    - ``Media_AEL['H2O(L)', 'H2', 'O2', 'H2O']`` - For AEL electrolyser systems
    - ``Steam['H2O]`` - Pure steam flows
    - ``Steam_Water['H2O', 'H2O(L)']`` - Two Phase water flows
    - ``Water['H2O(L)']`` - Pure liquid water flows

This approach allows for smaller systems of equations compared with loading all species.
This also allows for easy development of new media models based on different thermodynamic libraries i.e. cantera, coolprop etc.
The NASA properties data can be found in celeste/data. You can use the `Thermo.df` to see the different species and data conveniently as a pandas DataFrame.

In [6]:
from celeste.CSL.thermo import Thermo
print(f'Number of Species = {Thermo.df.index.size}')

Number of Species = 1899


Let's make our own media model for the Mixer:

In [7]:
class MediaExample(media.Medium_NASAbase):
    def __init__(self, name = 'MediaExample'):
        self._reference_X = np.array([0.25, 0.5, 0.25])  # needed for when you don't care about setting initial compositions
        super().__init__(name, species=np.array(['CO2', 'CO', 'CH4']) , Xi=self._reference_X)

Using this media in a flange would look like:

In [8]:
ExampleFlange = Flange('ExampleFlange', Medium=MediaExample()) # use defaults for n, Xi, T, P

In [9]:
print(ExampleFlange.Xi) # taken from reference_X
print(ExampleFlange.T, ExampleFlange.P )
print(ExampleFlange.h ) # Calculated at default conditions

[0.25 0.5  0.25]
298.15 1.0
-172.2941160036823


## Finishing Model

Now that we know how to use the ``Flange`` and ``media`` models, we can complete the ``__init__`` method by passing the necessary parameters in:

```python

class Mixer(ModelBase):
    def __init__(self, name, Medium, n_F0, Xi_F0, T_F0, P_F0, 
                                    n_F1, Xi_F1, T_F1, P_F1, 
                                    n_F2, Xi_F2, T_F2, P_F2):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, n=n_F0, Xi=Xi_F0, T=T_F0, P=P_F0)
        self.F1 = Flange(name + '.F1', Medium=Medium, n=n_F1, Xi=Xi_F1, T=T_F1, P=P_F1)
        self.F2 = Flange(name + '.F2', Medium=Medium, n=n_F2, Xi=Xi_F2, T=T_F2, P=P_F2)
```

Depending on the model, we can also include other variables that we deem constant or pre initialise some variables to avoid creating them later on for performance reasons.

We can then move on to including the system of equations needed to solve the model. These are usually done through balance equations of the form
`[Input] - [Output] + [Generation] - [Consumption] = 0`
- Starting with the mass balance: `F0 + F1 - F2 = 0`
- There is also a balance for the species: `X0_i*F0 + X1_i*F1 - X2_i*F2 = 0` for given component `i`
- Then the Energy balance: `h0*F0 + h1*F1 - h2*F2 = 0`
- Finally the pressure. Here we will simplify this to assign a pressure to the outlet stream based on F0: `P2 = P0`

This looks like the following:
```python

    def Eqs(self):
        self.Eq = np.concatenate((
            [self.F0.n + self.F1.n - self.F2.n],
            self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n - self.F2.Xi*self.F2.n,
            [self.F0.h*self.F0.n + self.F1.h*self.F1.n - self.F2.h*self.F2.n]
        ))
        self.F2.P = self.F0.P
        return self.Eq
```

Then we have to decide on the unknowns, which are `F2.n`, `F2.Xi`, and `F2.T` and we use this to build `f_sol`

```python

    def f_sol(self, x):
        self.F2.n = x[0]
        self.F2.Xi = x[1:self.Medium.nSpecies]
        self.F2.T = x[self.Medium.nSpecies]
        return self.Eqs() # update self.Eq based on x
```
Now we have formed the system of equations in a form that can be used by numerical solvers, where we give in a parameter `x` into `f_sol` and we get the errors of the system of equations from `self.Eq`. The solver will iteratively change `x`, until `self.Eq` is a vector of zeros (or close enough). In this case we wrap this functionality into the unit itself since we don't want to deal with giving guesses to all the different units and systems. So in this case we relate the initial guess to the input Flanges.

```python
    def run_sol(self):
        x0 = np.concatenate((
            [self.F0.n + self.F1.n],
            (self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n)/self.F2.n,
            self.F0.T
        ))
        sol = spo.root(self.f_sol, x0)
        return sol
```

For most cases we use `scipy.optimize.root()` for solving the system of equations. This library is usually imported as `import scipy.optimize as spo`. Another solver that is used in `spo` is `least_squares()` which is used in a similar way.

bringing it all together we get the following:

In [10]:
import scipy.optimize as spo

class Mixer(ModelBase):
    def __init__(self, name, Medium, n_F0, Xi_F0, T_F0, P_F0, 
                                    n_F1, Xi_F1, T_F1, P_F1, 
                                    n_F2, Xi_F2, T_F2, P_F2):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, n=n_F0, Xi=Xi_F0, T=T_F0, P=P_F0)
        self.F1 = Flange(name + '.F1', Medium=Medium, n=n_F1, Xi=Xi_F1, T=T_F1, P=P_F1)
        self.F2 = Flange(name + '.F2', Medium=Medium, n=n_F2, Xi=Xi_F2, T=T_F2, P=P_F2)

    def Eqs(self):
            self.Eq = np.concatenate((
                [self.F0.n + self.F1.n - self.F2.n],
                self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n - self.F2.Xi*self.F2.n,
                [self.F0.h*self.F0.n + self.F1.h*self.F1.n - self.F2.h*self.F2.n]
            ))
            self.F2.P = self.F0.P
            return self.Eq

    def f_sol(self, x):
            self.F2.n = x[0]
            self.F2.Xi = x[1:self.Medium.Ns+1]
            self.F2.T = x[self.Medium.Ns+1]
            return self.Eqs() 
        
    def run_sol(self):
        x0 = np.concatenate((
            [self.F0.n + self.F1.n],
            (self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n)/self.F2.n,
            [self.F0.T]
        ))
        sol = spo.root(self.f_sol, x0)
        return sol

Lets test this model:

In [11]:
mixer = Mixer('mixer', MediaExample(), n_F0 = 1, Xi_F0 = np.array([0, 1, 0]), T_F0 = 300, P_F0 = 1, 
              n_F1 = 1, Xi_F1 = np.array([1, 0,0]), T_F1 = 400, P_F1 = 1, 
              n_F2 = 1, Xi_F2 = None, T_F2 = 400, P_F2 = 1)

We can see the residulas based on the initial values we set:

In [12]:
mixer.Eqs()

array([ 1.00000000e+00,  7.50000000e-01,  5.00000000e-01, -2.50000000e-01,
       -3.31146952e+02])

And then running the system of equations:

In [13]:
mixer.run_sol()

 message: The solution converged.
 success: True
  status: 1
     fun: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00]
       x: [ 2.000e+00  5.000e-01  5.000e-01  0.000e+00  3.581e+02]
  method: hybr
    nfev: 14
    fjac: [[-1.984e-03 -1.984e-03 ...  0.000e+00  1.000e+00]
           [ 6.930e-01 -1.987e-01 ...  0.000e+00  2.356e-03]
           ...
           [ 0.000e+00  0.000e+00 ... -1.000e+00  0.000e+00]
           [-4.785e-03 -9.625e-01 ...  0.000e+00 -2.458e-03]]
       r: [ 5.039e+02  7.833e+02 ...  0.000e+00  1.694e-04]
     qtf: [ 1.435e-09  3.381e-12  6.765e-13  0.000e+00 -3.527e-12]

We see that the system solved successfully!
Now let's confirm the results make sense by looking into the outlet Flange

In [14]:
mixer.F2

n /mol/s          2.000000
m /g/s           72.019600
T /K            358.144283
P /bar            1.000000
h /kJ/mol      -249.992395
cp /kJ/mol/K      0.034472
Xi_CO2            0.500000
Xi_CO             0.500000
Xi_CH4            0.000000

We can store this model in our own library to reuse the component without copy pasting everytime

## Building Your Own Component Library

It is not recommended to alter the code in the CSL. Instead, if you want to develop your own models, this should be done locally.
If you then want these models to be shared through the CSL, you can do this through the git issues.
Here we will discuss how you can build your own library and use them in your code.

First off, you need to make the folders where these components will be stored. This can be anywhere in your PC, but keep note of it. Here we will refer to this folder as `mylibrary`

So your folder structure would look like:
```tree
├──[My-CELESTE-Projects]
|    ├──mylibrary
|    |   ├── __init__.py
|    |   ├── mycomponents.py
|    |
|    ├──[Project Folder 1] 
|    ├──[Project Folder 2] 
|    ├──[Project Folder ...]
|    ├──[Project Folder N]
|
└──[Other Files]
```

The `__init__.py` is just an empty text file and is needed to tell python that `mylibrary` is a python module and all files ending in `.py` are submodules that can be imported in other python programmes.

Once you have these folders, you can paste your CELESTE objects into `mycomponents.py` along with any imports needed. So `mycomponents.py` should look like the following:


```python

import scipy.optimize as spo
from celeste.CSL.interfaces import Flange
import celeste.CSL.media as media
import numpy as np

class MediaExample(media.Medium_NASAbase):
    def __init__(self, name = 'MediaExample'):
        self._reference_X = np.array([0.25, 0.5, 0.25])  # needed for when you don't care about setting initial compositions
        super().__init__(name, species=np.array(['CO2', 'CO', 'CH4']) , Xi=self._reference_X)

class Mixer(ModelBase):
    def __init__(self, name, Medium, n_F0, Xi_F0, T_F0, P_F0, 
                                    n_F1, Xi_F1, T_F1, P_F1, 
                                    n_F2, Xi_F2, T_F2, P_F2):
        super().__init__(locals())
        self.F0 = Flange(name + '.F0', Medium=Medium, n=n_F0, Xi=Xi_F0, T=T_F0, P=P_F0)
        self.F1 = Flange(name + '.F1', Medium=Medium, n=n_F1, Xi=Xi_F1, T=T_F1, P=P_F1)
        self.F2 = Flange(name + '.F2', Medium=Medium, n=n_F2, Xi=Xi_F2, T=T_F2, P=P_F2)

    def Eqs(self):
            self.Eq = np.concatenate((
                [self.F0.n + self.F1.n - self.F2.n],
                self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n - self.F2.Xi*self.F2.n,
                [self.F0.h*self.F0.n + self.F1.h*self.F1.n - self.F2.h*self.F2.n]
            ))
            self.F2.P = self.F0.P
            return self.Eq

    def f_sol(self, x):
            self.F2.n = x[0]
            self.F2.Xi = x[1:self.Medium.Ns+1]
            self.F2.T = x[self.Medium.Ns+1]
            return self.Eqs() 
        
    def run_sol(self):
        x0 = np.concatenate((
            [self.F0.n + self.F1.n],
            (self.F0.Xi*self.F0.n + self.F1.Xi*self.F1.n)/self.F2.n,
            [self.F0.T]
        ))
        sol = spo.root(self.f_sol, x0)
        return sol

```

To be able to import this code into your projects, Python needs to know where this folder is. There are two approaches we can take to achieve this:
1. Using a .pth file
2. Making a pip installable package of your component library

Both approaches allow you to continuously build your library of components.

In the following setion we will outline both ways.

### .pth File Approach

1. Create a text file and change the extension from `.txt` to ``.pth`` i.e. `mycelestelibrary.pth` (like Tutorial/Example.pth)
2. In this file paste in the complete path to `[My-CELESTE-Projects]`

[Python]
3. Now open a command prompt and write `where python` and open the folder this `python.exe` file exists in.

[conda]
3. Open conda prompt and write `conda env list`. Open the foler of the environment you want to work in

4. Go into the folders Lib/site-packages/
5. Paste the `mycelestelibrary.pth`
6. Test that this works by opening a python terminal (open cmd or conda prompt and type `python` and enter) then `import mylibrary` you should see no error.

### Pip Installable Package Approach

1. Ccopy the two text files, `pyproject.toml`(i.e. Tutorial\example_pyproject.toml) and `setup.cfg`(i.e. Tutorial\example_setup.cfg) and paste into the root of `[My-CELESTE-Projects]`
2. In `setup.cfg` fill in the necessary data
3. Open your python/conda terminal and `cd [path to MY-CELESTE-projects]`
4. then pip install -e . to perform an editable install of your library.
5. If everything went well you should see success.
6. To test, in the terminal open python then `import mylibrary`. You should see no errors

# Part 3 - Systems

For this section we will use the components in the CSL. But the procedure is the same with your own library.

We will use the folder tree structure already established and work in the folder `Project Folder 1`

Here we will make a file called `projectsystem.py` and a jupyter notebook called `runprojectsystem.ipynb`.

The `.py` file will contain our system which extends from `celeste.CSL.SysBase` and the `.ipynb` file will run and explore the results and make plots etc.

`celeste.CSL.SysBase` contains some base code that is inherited by other systems including methods to run recycles, make graphs and connect unit objects both acausally (`connect()`), and sequentially (`connect_seq()`). In our case we will go with the sequential approach.

in the `.py` file we will make a system model that looks like the following

```python
from celeste.CSL import SysBase
class ProjSys(SysBase):
    def __init__(self, [parameters]):
        super().__init__(locals())
        #Initialise components with parameters
        self.source1 = Source(params)
        self.component1 = Component1(Params) 
        self.component1 = Component1(Params) 
        self.sink1 = Sink(params)
        [Other components]

    def seq_sol(self, store=False): # Sequentially flow through PFD
        self.connect_seq(self.source1.F1, self.component1.F0, store=store) # Update feed of component1
        self.component1._sol = self.component1.run_sol() # Solve component1 based on update and store solver results in model
        self.connect_seq(self.component1.F1, self.component2.F0, store=store) # Update component2
        self.component2._sol = self.component2.run_sol() # Solve component2 based on update and store solver results in model
        self.connect_seq(self.component2.F1, self.snk.F0, store=store)
        [connect and solve other components]

    def Optimiser(self, x):
        [components to be varied] = x

        self.seq_sol() # update the pfd based on given x, could also be done with recycle

        X = np.zeros(N) # N number of equations or conditions to optimise
        X[0] = [Condition 1]
        X[...] = [Conditions ...]
        X[N-1] = [Condition N]

        return X
```

We will use this template to build our own simple system model for the following PFD

In [16]:
import numpy as np

from celeste.CSL import flow, components, reactors, interfaces, media, SysBase
class ExampleReformerSys(SysBase):
    def __init__(self, name, n0, X0, T, p):
        super().__init__(locals())
        #Initialise components with parameters
        self.source1 = flow.SourceM('source1', media.Fuel_CH4(), n0, X0, T, p)
        self.reformer = reactors.MR('reformer', media.Fuel_CH4(), T=T)
        self.sink1 = flow.Sink('sink1', media.Fuel_CH4())

    def seq_sol(self, store=False): # Sequentially flow forward through PFD
        self.connect_seq(self.source1.F1, self.reformer.F0, store=store) # Update feed of component1
        self.reformer._sol = self.reformer.run_sol() # Solve component1 based on update and store solver results in model
        self.connect_seq(self.reformer.F1, self.sink1.F0, store=store) # update sink

    def Optimiser(self, x):
        '''
        Alter the inlet flow until a desired outlet flow of the reformer is achieved

        Arguments:
            x [np.Array] : guesses for solver

        Returns:
            X [np.Array] : Residual of system of equations
        '''
        self.source1.F1.n = x[0]

        self.seq_sol() # update the system based on given x, could also be done with recycle

        X = np.zeros(1) # N number of equations or conditions to optimise
        X[0] = self.sink1.F0.n - 2 # we want the outflow flow rate to be 2 mol/s
        return X

Now to use this model we create an instance

In [17]:
reformerSys = ExampleReformerSys('ReformerSys', 1, np.array([0.01, 0.3, 0.2, 1e-5, 0.5, 1e-5]), 973.15, 5.)

At this point the system is unsolved so if we were to see the flanges, the flow sheet would not make much sense:

In [18]:
reformerSys.ViewFlange([reformerSys.source1.F1, reformerSys.reformer.F0, reformerSys.reformer.F1, reformerSys.sink1.F0])

,source1((source1)).F1,reformer.F0,reformer.F1,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000e+00,1.000000e+00,1.000000e+00
m /g/s,22.642997,1.641534e+01,1.641534e+01,1.641534e+01
T /K,973.150000,9.731500e+02,9.731500e+02,2.981500e+02
P /bar,5.000000,5.000000e+00,5.000000e+00,1.000000e+00
h /kJ/mol,-191.940338,-1.932459e+02,-1.932459e+02,-2.176422e+02
cp /kJ/mol/K,0.053314,3.985893e-02,3.985893e-02,3.311239e-02
Xi_H2,0.010000,1.000000e-01,1.000000e-01,1.000000e-01
Xi_CH4,0.300000,1.000000e-08,1.000000e-08,1.000000e-08
Xi_CO2,0.200000,1.000000e-08,1.000000e-08,1.000000e-08
Xi_CO,0.000010,1.000000e-08,1.000000e-08,1.000000e-08


So let's run this system for the forward pass

In [19]:
reformerSys.seq_sol()

In [20]:
reformerSys.ViewFlange([reformerSys.source1.F1, reformerSys.reformer.F0, reformerSys.reformer.F1, reformerSys.sink1.F0])

,source1((source1)).F1,reformer.F0,reformer.F1,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000,1.386519,1.386519
m /g/s,22.642997,22.642997,22.642997,22.642997
T /K,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.624925,-107.624925
cp /kJ/mol/K,0.053314,0.053314,0.039709,0.039709
Xi_H2,0.010000,0.010000,0.405848,0.405848
Xi_CH4,0.300000,0.300000,0.080598,0.080598
Xi_CO2,0.200000,0.200000,0.135568,0.135568
Xi_CO,0.000010,0.000010,0.144456,0.144456


we can verify that the system solved properly in two ways:
1) check the mass balance of a process unit if the `MBcheck()` method exists, or 
2) Directly look at the solver results if they exist for the unit with `UNIT._sol`

In [21]:
reformerSys.reformer.MBcheck()

1.1102230246251565e-14% difference 


np.float64(-3.552713678800501e-15)

In [22]:
reformerSys.reformer._sol

     message: `gtol` termination condition is satisfied.
     success: True
      status: 1
         fun: [ 0.000e+00  4.441e-16  0.000e+00  3.388e-21  0.000e+00
               -1.388e-17 -8.327e-17]
           x: [ 1.387e+00  4.058e-01  8.060e-02  1.356e-01  1.445e-01
                2.335e-01  7.212e-06]
        cost: 1.0217058355548258e-31
         jac: [[-3.606e-01  0.000e+00 ...  0.000e+00  0.000e+00]
               [-1.601e+00 -2.773e+00 ... -2.773e+00  0.000e+00]
               ...
               [ 0.000e+00 -1.356e-01 ...  2.356e-01  0.000e+00]
               [ 0.000e+00 -1.785e+00 ...  1.034e+00  0.000e+00]]
        grad: [-7.110e-16 -1.081e-15 -2.712e-15  5.632e-18  1.339e-16
               -1.321e-15 -9.395e-21]
  optimality: 2.7123625777305885e-15
 active_mask: [ 0.000e+00  0.000e+00  0.000e+00  0.000e+00  0.000e+00
                0.000e+00  0.000e+00]
        nfev: 6
        njev: 6

In both cases the results show the system solved successfully.

## makeGraph

Now lets briefly go back to the `.seq_sol()` method. We have an additional argument `store`

This is useful for generating the PFD of the system as a dophin.JS graph with the method `.makeGraph()`. Let's give it a try

In [34]:
reformerSys.makeGraph()

this generates a `.mmd` file with the following content

which when executed in a markdown executor that can evaluate mermaidJS (such as jupyter notebooks ;) ) you get the following

```mermaid
flowchart LR
source1-- F1_F0 -->reformer
reformer-- F1_F0 -->sink1
```

Notice that the name of the units is the same as that of the system attibutes - this is because the name's of the units in the strings was purposely made the same as the object in python itself - do this! it will help with any future debugging
- TL;DR - give your process units the same python name as the string i.e. `self.source1 = flow.SourceM('source1', ...)` and NOT `self.source1 = flow.SourceM('Source', ...)` 

## Optimisation
Now we have a small system, lets use it to do some optimisation. In this case, we don't want to to solve the nonlinear systems in the back and just want the model to tell us what inlet flowrate is needed to get a desired outlet flow of 2mol/s (pretty simple example, but other examples of nonlinear optimisation/inverse problems follow the same principle)

Let's use the optimiser function we built with the system to do this.

It should be noted that the function we built, isn't the actual optimiser, it is a function needed by the optimiser to iteratively run and make necessary modifications until a certain tolerance is met (or until it fails). In CELESTE we mostly use Scipy.Optimize - since it is written in the most flexible way and has a wide variety of optimisers available to use.

In [23]:
import scipy.optimize as spo

sol = spo.root(reformerSys.Optimiser, reformerSys.source1.F1.n)

In [24]:
sol

 message: The solution converged.
 success: True
  status: 1
     fun: [ 4.441e-16]
       x: [ 1.442e+00]
  method: hybr
    nfev: 6
    fjac: [[-1.000e+00]]
       r: [-1.387e+00]
     qtf: [ 4.315e-09]

### Okay... so what did we do here?

pretty simple:
1. We use the `root` optimiser from scipy.optimize (more details in their [documentation]('https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root.html')) (Also relevant for other solvers such as `least_squares`, `minimize`, ... and methods/algorithms)
2. We told the solver to use the Optimiser method in reformerSys with the initial guess for the solution as the current value for the inlet flow rate
3. from looking into sol, we see that our solution converged with very high accuracy and very quickly!

Let's see what this looks like in our flanges

In [25]:
reformerSys.ViewFlange([reformerSys.source1.F1, reformerSys.reformer.F0, reformerSys.reformer.F1, reformerSys.sink1.F0])

,source1((source1)).F1,reformer.F0,reformer.F1,sink1[(sink1)].F0
n /mol/s,1.442462,1.442462,2.000000,2.000000
m /g/s,32.661652,32.661652,32.661652,32.661652
T /K,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.624925,-107.624925
cp /kJ/mol/K,0.053314,0.053314,0.039709,0.039709
Xi_H2,0.010000,0.010000,0.405848,0.405848
Xi_CH4,0.300000,0.300000,0.080598,0.080598
Xi_CO2,0.200000,0.200000,0.135568,0.135568
Xi_CO,0.000010,0.000010,0.144456,0.144456


So clearly we got what we wanted!

## What about recycles?

We can modify our system to consider recyles very easily. SysBase already has methods for how to operate recycles, we just need to tell it what streams need to be recycled. From an engineering point of view, this of course means that we need to add splitters and mixers to realise the recycle. So let's make these changes to the model.

In [26]:
import numpy as np

from celeste.CSL import flow, components, reactors, interfaces, media, SysBase
class ExampleReformerRecycleSys(SysBase):
    def __init__(self, name, n0, X0, T, p):
        super().__init__(locals())
        #Initialise components with parameters
        self.source1 = flow.SourceM('source1', media.Fuel_CH4(), n0, X0, T, p)
        self.reformer = reactors.MR('reformer', media.Fuel_CH4(), T=T)
        self.sink1 = flow.Sink('sink1', media.Fuel_CH4())
        self.split = flow.Splitter('split', media.Fuel_CH4(), Split=0.5)
        self.mix = flow.Mixer('mix', media.Fuel_CH4())
        self.mix.F1.n = 0                                               # no recycle on the first pass

    def seq_sol(self, store=False): # Sequentially flow forward through PFD
        self.connect_seq(self.source1.F1, self.mix.F0, store=store)     # feed to the mixer
        self.mix._sol = self.mix.run_sol()                              # Solve mixer - mix.F1 will be given by recycle
        self.connect_seq(self.mix.F2, self.reformer.F0, store=store)    # outlet of mixer to reformer
        self.reformer._sol = self.reformer.run_sol()                    # Solve reformer
        self.connect_seq(self.reformer.F1, self.split.F0, store=store)  # Feed to splitter
        self.split._sol = self.split.run_sol()                          # update splitter
        self.connect_seq(self.split.F1, self.sink1.F0, store=store)  # update sink

    def Optimiser(self, x):
        '''
        Alter the inlet flow until a desired outlet flow of the reformer is achieved

        Arguments:
            x [np.Array] : guesses for solver

        Returns:
            X [np.Array] : Residual of system of equations
        '''
        self.source1.F1.n = x[0]

        self.Recycle_Tear(self.split.F2, self.mix.F1) # update the system based on given x with the recycle from splitter to mixer
                                        # Wegstein_Generic algorithm used for recycling

        X = np.zeros(1) # N number of equations or conditions to optimise
        X[0] = self.sink1.F0.n - 2 # we want the outflow flow rate to be 2 mol/s
        return X

In [27]:
reformerSysRecycle = ExampleReformerRecycleSys('ReformerSys', 1, np.array([0.01, 0.3, 0.2, 1e-5, 0.5, 1e-5]), 973.15, 5.)

Let's have a look at the graph

In [28]:
reformerSysRecycle.makeGraph()

Let's do a forward pass of this system without a recycle

In [29]:
reformerSysRecycle.seq_sol()

```mermaid
flowchart LR
source1-- F1_F0 -->mix
mix-- F2_F0 -->reformer
reformer-- F1_F0 -->split
split-- F1_F0 -->sink1
```

Makes sense we didn't explicitly write the recycle in the forward pass (otherwise it would slow down the recycle algorithm)

Now let's see the results of the model

In [30]:
reformerSysRecycle.ViewFlange([reformerSysRecycle.source1.F1, 
                                reformerSysRecycle.mix.F0,
                                reformerSysRecycle.mix.F1,
                                reformerSysRecycle.mix.F2,
                                reformerSysRecycle.reformer.F0, 
                                reformerSysRecycle.reformer.F1, 
                                reformerSysRecycle.split.F0,
                                reformerSysRecycle.split.F1,
                                reformerSysRecycle.split.F2,
                                reformerSysRecycle.sink1.F0])

,source1((source1)).F1,mix.F0,mix.F1,mix.F2,reformer.F0,reformer.F1,split.F0,split.F1,split.F2,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000,0.000000e+00,1.000000,1.000000,1.386519,1.386519,0.693259,0.693259,0.693259
m /g/s,22.642997,22.642997,0.000000e+00,22.642997,22.642997,22.642997,22.642997,11.321499,11.321499,11.321499
T /K,973.150000,973.150000,2.981500e+02,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,1.000000e+00,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-2.176422e+02,-191.940338,-191.940338,-107.624925,-107.624925,-107.624925,-107.624925,-107.624925
cp /kJ/mol/K,0.053314,0.053314,3.311239e-02,0.053314,0.053314,0.039709,0.039709,0.039709,0.039709,0.039709
Xi_H2,0.010000,0.010000,1.000000e-01,0.010000,0.010000,0.405848,0.405848,0.405848,0.405848,0.405848
Xi_CH4,0.300000,0.300000,1.000000e-08,0.300000,0.300000,0.080598,0.080598,0.080598,0.080598,0.080598
Xi_CO2,0.200000,0.200000,1.000000e-08,0.200000,0.200000,0.135568,0.135568,0.135568,0.135568,0.135568
Xi_CO,0.000010,0.000010,1.000000e-08,0.000010,0.000010,0.144456,0.144456,0.144456,0.144456,0.144456


Already looking different to our previous system. Let's run the recycle if that helps since our mix.F1 has a near 0 flow rate

In [31]:
reformerSysRecycle.Recycle_Tear(reformerSysRecycle.split.F2, reformerSysRecycle.mix.F1)

(2, np.float64(2.962689932421661e-14))

In [32]:
reformerSysRecycle.ViewFlange([reformerSysRecycle.source1.F1, 
                                reformerSysRecycle.mix.F0,
                                reformerSysRecycle.mix.F1,
                                reformerSysRecycle.mix.F2,
                                reformerSysRecycle.reformer.F0, 
                                reformerSysRecycle.reformer.F1, 
                                reformerSysRecycle.split.F0,
                                reformerSysRecycle.split.F1,
                                reformerSysRecycle.split.F2,
                                reformerSysRecycle.sink1.F0])

,source1((source1)).F1,mix.F0,mix.F1,mix.F2,reformer.F0,reformer.F1,split.F0,split.F1,split.F2,sink1[(sink1)].F0
n /mol/s,1.000000,1.000000,1.386519,2.386519,2.386519,2.773038,2.773038,1.386519,1.386519,1.386519
m /g/s,22.642997,22.642997,22.642997,45.285994,45.285994,45.285994,45.285994,22.642997,22.642997,22.642997
T /K,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.624925,-142.954801,-142.954801,-107.624925,-107.624925,-107.624925,-107.624925,-107.624925
cp /kJ/mol/K,0.053314,0.053314,0.039709,0.045410,0.045410,0.039709,0.039709,0.039709,0.039709,0.039709
Xi_H2,0.010000,0.010000,0.405848,0.239980,0.239980,0.405848,0.405848,0.405848,0.405848,0.405848
Xi_CH4,0.300000,0.300000,0.080598,0.172532,0.172532,0.080598,0.080598,0.080598,0.080598,0.080598
Xi_CO2,0.200000,0.200000,0.135568,0.162566,0.162566,0.135568,0.135568,0.135568,0.135568,0.135568
Xi_CO,0.000010,0.000010,0.144456,0.083930,0.083930,0.144456,0.144456,0.144456,0.144456,0.144456


It was able to solve the recycle very quickly, with very few iterations, now let's see if it can handle optimisation.

In [36]:
sol_rec = spo.root(reformerSysRecycle.Optimiser, reformerSysRecycle.source1.F1.n)
sol_rec

 message: The solution converged.
 success: True
  status: 1
     fun: [ 6.760e-10]
       x: [ 1.442e+00]
  method: hybr
    nfev: 6
    fjac: [[-1.000e+00]]
       r: [-1.501e+00]
     qtf: [-5.589e-09]

In [37]:
reformerSysRecycle.ViewFlange([reformerSysRecycle.source1.F1, 
                                reformerSysRecycle.mix.F0,
                                reformerSysRecycle.mix.F1,
                                reformerSysRecycle.mix.F2,
                                reformerSysRecycle.reformer.F0, 
                                reformerSysRecycle.reformer.F1, 
                                reformerSysRecycle.split.F0,
                                reformerSysRecycle.split.F1,
                                reformerSysRecycle.split.F2,
                                reformerSysRecycle.sink1.F0])

,source1((source1)).F1,mix.F0,mix.F1,mix.F2,reformer.F0,reformer.F1,split.F0,split.F1,split.F2,sink1[(sink1)].F0
n /mol/s,1.442462,1.442462,2.000000,3.442462,3.442462,4.000000,4.000000,2.000000,2.000000,2.000000
m /g/s,32.661652,32.661652,32.661652,65.323305,65.323305,65.323305,65.323305,32.661652,32.661652,32.661652
T /K,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000,973.150000
P /bar,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
h /kJ/mol,-191.940338,-191.940338,-107.624925,-142.954801,-142.954801,-107.624925,-107.624925,-107.624925,-107.624925,-107.624925
cp /kJ/mol/K,0.053314,0.053314,0.039709,0.045410,0.045410,0.039709,0.039709,0.039709,0.039709,0.039709
Xi_H2,0.010000,0.010000,0.405848,0.239980,0.239980,0.405848,0.405848,0.405848,0.405848,0.405848
Xi_CH4,0.300000,0.300000,0.080598,0.172532,0.172532,0.080598,0.080598,0.080598,0.080598,0.080598
Xi_CO2,0.200000,0.200000,0.135568,0.162566,0.162566,0.135568,0.135568,0.135568,0.135568,0.135568
Xi_CO,0.000010,0.000010,0.144456,0.083930,0.083930,0.144456,0.144456,0.144456,0.144456,0.144456


Even though it says the solver failed, we can see that it actually succeeded since the residual of our function is almost 0 (see `fun` in the output of `sol_rec`) - this can happen sometimes, and it helps to use lower tolerances for larger systems where very high levels of accuracy are not needed.

# Fin.

With this you should be equipped enough to start playing around with your own models and understand the models we have already made for the units and systems (or to ask us for help with your models)